In [22]:
import os
from usleep_api import USleepAPI
import logging
from pathlib import Path
import time

In [23]:
binary_path = "D:/EEG_Data_stage/"
api_token = ''
epoch_length = 10
sampling_rate = 250
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("api")

In [ ]:
logger = logging.getLogger(__name__)
BACKOFF_FACTOR = 5  # seconds, for exponential backoff

for folder in os.listdir(binary_path):
    if folder not in ["line_per_state", "bandpower", "plots"]:
        subject_data = os.path.join(binary_path, folder, "iEEG", "converted_ec", "edf_files")
        output_dir = os.path.join(binary_path, folder, "iEEG", "U_sleep_API")
        os.makedirs(output_dir, exist_ok=True)
    
        for file in os.listdir(subject_data):
            file_stem = Path(file).stem
            output_file = Path(output_dir) / f"{file_stem}_hypnogram.npy"
    
            # ✅ Skip file if hypnogram already exists
            if output_file.exists():
                print(f"Skipping {file_stem} — hypnogram already exists.")
                continue
    
            full_path = Path(os.path.join(subject_data, file))
            print(f"\nScoring file: {file}")
    
            # Start a fresh session for each file
            api = USleepAPI(api_token=api_token)
            session = api.new_session(session_name="intra_scoring")
            session.set_model("U-Sleep v2.0")
    
            # Retry upload indefinitely until successful
            attempt = 1
            while True:
                try:
                    response = session.upload_file(full_path, anonymize_before_upload=False)
                    if hasattr(response, "status_code") and response.status_code == 504:
                        raise Exception("504 Gateway Timeout during upload.")
                    print("Upload response:", response)
                    break  # success
                except Exception as e:
                    logger.warning(f"Upload failed (attempt {attempt}): {e}")
                    time.sleep(BACKOFF_FACTOR * attempt)
                    attempt += 1
                    # 🔁 Reset session after failed upload to prevent 404 issues
                    session.delete_session()
                    session = api.new_session(session_name="intra_scoring")
                    session.set_model("U-Sleep v2.0")
    
            # Retry prediction indefinitely until successful
            attempt = 1
            while True:
                try:
                    pred_response = session.predict(
                        data_per_prediction=3840,
                        channel_groups=[
                            ["T5-Cz", "EOG1"],
                            ["T6-Cz", "EOG2"],
                            ["C3-Cz", "EOG1"],
                            ["C4-Cz", "EOG2"],
                            ["Oz-Cz", "EOG1"],
                        ],
                    )
    
                    print("Prediction response:", pred_response)
                    if hasattr(pred_response, "status_code") and pred_response.status_code == 504:
                        raise Exception("504 Gateway Timeout during prediction.")
                    elif hasattr(pred_response, "status_code") and pred_response.status_code == 404:
                        raise Exception("404: No valid file uploaded to session.")
    
                    success = session.wait_for_completion()
                    if success:
                        hyp = session.get_hypnogram()
                        logger.info(hyp.get("hypnogram", "No hypnogram data returned"))
                        session.download_hypnogram(out_path=output_file, file_type="npy")
                        print(f"Downloaded hypnogram for {file_stem}")
                        break  # success
                    else:
                        logger.error("Prediction failed — retrying...")
                        raise Exception("Prediction did not complete successfully")
    
                except Exception as e:
                    logger.warning(f"Prediction failed (attempt {attempt}): {e}")
                    time.sleep(BACKOFF_FACTOR * attempt)
                    attempt += 1
                    # 🔁 Reset session after persistent failures
                    session.delete_session()
                    session = api.new_session(session_name="intra_scoring")
                    session.set_model("U-Sleep v2.0")
                    # Re-upload the file in the new session
                    while True:
                        try:
                            response = session.upload_file(full_path, anonymize_before_upload=False)
                            if hasattr(response, "status_code") and response.status_code == 504:
                                raise Exception("504 Gateway Timeout during re-upload.")
                            break
                        except Exception as e2:
                            logger.warning(f"Re-upload failed (attempt {attempt}): {e2}")
                            time.sleep(BACKOFF_FACTOR * attempt)
                            attempt += 1
    
            session.delete_session()

Skipping 67_night1_01 — hypnogram already exists.
Skipping 67_night1_02 — hypnogram already exists.
Skipping 67_night1_03 — hypnogram already exists.
Skipping 67_night1_04 — hypnogram already exists.
Skipping 67_night2_01 — hypnogram already exists.
Skipping 67_night2_02 — hypnogram already exists.
Skipping 67_night2_03 — hypnogram already exists.
Skipping 67_night2_04 — hypnogram already exists.
Skipping 67_night2_05 — hypnogram already exists.
Skipping 67_paradigm — hypnogram already exists.


INFO:usleep_api.usleep_api:Validating auth token...



Scoring file: 69_night1_01.edf


INFO:usleep_api.usleep_api:Server response to GET: pong
INFO:usleep_api.usleep_api:Setting model 'U-Sleep v2.0'
INFO:usleep_api.usleep_api:Server response to GET: [JSON data] {'models': ['U-Sleep v1.0', 'U-Sleep v2.0', 'U-Sle ...
INFO:usleep_api.usleep_api:Server response to POST: New model 'U-Sleep v2.0' selected.
INFO:usleep_api.usleep_api:Uploading file at path D:\EEG_Data_stage\69\iEEG\converted_ec\edf_files\69_night1_01.edf. Please wait.
INFO:usleep_api.usleep_api:Server response to POST: New file uploaded.


Upload response: <Response [201]>


...

Downloaded hypnogram for 69_night2_01

Scoring file: 69_night2_02.edf


INFO:usleep_api.usleep_api:Server response to GET: pong
INFO:usleep_api.usleep_api:Setting model 'U-Sleep v2.0'
INFO:usleep_api.usleep_api:Server response to GET: [JSON data] {'models': ['U-Sleep v1.0', 'U-Sleep v2.0', 'U-Sle ...
INFO:usleep_api.usleep_api:Server response to POST: New model 'U-Sleep v2.0' selected.
INFO:usleep_api.usleep_api:Uploading file at path D:\EEG_Data_stage\69\iEEG\converted_ec\edf_files\69_night2_02.edf. Please wait.
INFO:usleep_api.usleep_api:Server response to POST: New file uploaded.
INFO:usleep_api.usleep_api:Server response to POST: Prediction started.


Upload response: <Response [201]>


INFO:usleep_api.usleep_api:Waiting for prediction completion...


Prediction response: <Response [201]>
